In [4]:
pip install pandas

   ---------------------------------------- 0.0/11.5 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.5 MB 4.0 MB/s eta 0:00:03
   ---- ----------------------------------- 1.3/11.5 MB 3.4 MB/s eta 0:00:04
   -------- ------------------------------- 2.4/11.5 MB 4.0 MB/s eta 0:00:03
   ----------- ---------------------------- 3.4/11.5 MB 4.3 MB/s eta 0:00:02
   -------------- ------------------------- 4.2/11.5 MB 4.2 MB/s eta 0:00:02
   ------------------ --------------------- 5.2/11.5 MB 4.3 MB/s eta 0:00:02
   --------------------- ------------------ 6.0/11.5 MB 4.2 MB/s eta 0:00:02
   ----------------------- ---------------- 6.8/11.5 MB 4.2 MB/s eta 0:00:02
   --------------------------- ------------ 7.9/11.5 MB 4.2 MB/s eta 0:00:01
   ------------------------------ --------- 8.7/11.5 MB 4.2 MB/s eta 0:00:01
   -------------------------------- ------- 9.4/11.5 MB 4.2 MB/s eta 0:00:01
   ----------------------------------- ---- 10.2/11.5 MB 4.2 MB/s eta 0:00:01
   --

In [16]:
import pandas as pd
from sqlalchemy import create_engine

# Connection details (replace with your actual credentials)  
server = r'HITESH-SAWLANI2\SQLEXPRESS'  # Use raw string to avoid escape sequence issues
database = 'AdventureWorksDW2019'  
driver = 'ODBC Driver 17 for SQL Server'  # Ensure the driver is correctly quoted

# Create the connection string for SQLAlchemy
connection_string = f'mssql+pyodbc://{server}/{database}?driver={driver}'

try:  
    # Create the SQLAlchemy engine
    engine = create_engine(connection_string)

    sql_query = """
    SELECT 
        d.EnglishCountryRegionName AS Region,
        p.EnglishProductName AS Product,
        c.FirstName + ' ' + c.LastName AS Customer,
        f.SalesAmount, 
        f.OrderDate
    FROM FactInternetSales f
    JOIN DimCustomer c ON f.CustomerKey = c.CustomerKey
    JOIN DimProduct p ON f.ProductKey = p.ProductKey
    JOIN DimGeography d ON c.GeographyKey = d.GeographyKey
    """

    # Fetch data into a DataFrame using SQLAlchemy
    df = pd.read_sql(sql_query, engine)  
    print(df.head())  # Display the first few rows of the DataFrame

    # Export to CSV using pandas  
    csv_file_path = r"F:\MARKET ANALYSIS\dbo.AdventureWorksDWBuildVersion.csv"  # Use raw string for file path
    df.to_csv(csv_file_path, index=False)  # index=False prevents writing row indices to the CSV  

    print(f"Data exported to {csv_file_path}")  

except Exception as ex:  
    print(f"An error occurred: {ex}")  
finally:  
    # No need to close the engine; SQLAlchemy handles it
    print("Process completed.")

          Region                  Product          Customer  SalesAmount  \
0         Canada         Road-150 Red, 62       Cole Watson    3578.2700   
1         France  Mountain-100 Silver, 44  Rachael Martinez    3399.9900   
2  United States  Mountain-100 Silver, 44     Sydney Wright    3399.9900   
3  United States       Road-650 Black, 62      Ruben Prasad     699.0982   
4      Australia  Mountain-100 Silver, 44       Christy Zhu    3399.9900   

   OrderDate  
0 2010-12-29  
1 2010-12-29  
2 2010-12-29  
3 2010-12-29  
4 2010-12-29  
Data exported to F:\MARKET ANALYSIS\dbo.AdventureWorksDWBuildVersion.csv
Process completed.


In [18]:
# Handle dates
df['OrderDate'] = pd.to_datetime(df['OrderDate'])
df['OrderMonth'] = df['OrderDate'].dt.to_period('M')

# Create new features
df['SalesCategory'] = pd.cut(df['SalesAmount'], bins=[0,100,500,1000,10000],
                             labels=["Low", "Medium", "High", "Very High"])


In [21]:
df.to_excel("Cleaned_Sales_Data.xlsx", index=False)